# 04. KuaiRec EU-MLP Pretraining for KuaiRand Transfer

This notebook trains an EU predictor on KuaiRec only. It uses the same KuaiRec tensor package as notebook 03 and restricts the feature set to the columns that notebook 02 marked as available for KuaiRand transfer.

It does not load KuaiRand interaction rows and does not score KuaiRand random recommendations. The only KuaiRand-side artifact read here is the feature manifest, which supplies the transferable feature contract.

## Data, Label, and Feature Contract

Training data: KuaiRec `gnn_data.pt` edge features.

Prediction label: posterior expected utility `EU` from notebook 07, aligned by `row_id`/edge index through `gnn_eu_targets.pt`.

Feature set: `manifest['prediction_feature_cols']`, exactly the KuaiRand-compatible feature list used by notebook 03 after dropping tag fields during feature construction. Continuous features, binary features, and categorical embeddings follow the same representation as notebook 03.

## Training Controls

The model is a scalar-output MLP. The EU target is standardized using the training split mean and standard deviation for numerical stability, but metrics are reported on the original EU scale. Early stopping is controlled by `NUM_EPOCHS`, `EARLY_STOPPING_PATIENCE`, and `MIN_VALID_LOSS_DELTA`.

In [1]:
from pathlib import Path
import copy
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
KUAIRAND_DIR = PROJECT_ROOT / "python_code_KuaiRand"
KUAIREC_PROCESSED_DIR = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"
STRUCTURAL_OUTPUT_DIR = PROJECT_ROOT / "python_code_new" / "outputs"

USE_TINY = False
GNN_DATA_PATH = KUAIREC_PROCESSED_DIR / ("gnn_data_tiny.pt" if USE_TINY else "gnn_data.pt")
KUAIREC_SOURCE_TABLE_PATH = KUAIREC_PROCESSED_DIR / "processed_data.parquet"
EU_TARGET_TENSOR_PATH = STRUCTURAL_OUTPUT_DIR / "gnn_eu_targets.pt"
EU_TARGET_TABLE_PATH = STRUCTURAL_OUTPUT_DIR / "gnn_eu_targets.parquet"
KUAIRAND_FEATURE_MANIFEST_PATH = KUAIRAND_DIR / "outputs" / "feature_build" / "kuairand_random_prediction_feature_manifest.json"

OUTPUT_DIR = KUAIRAND_DIR / "model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "kuairec_transfer_eu_mlp_kuairand_features"
MODEL_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}.pt"
METRICS_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_metrics.json"
HISTORY_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_history.csv"
FEATURE_SPEC_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_feature_spec.json"

RANDOM_SEED = 20260706
BATCH_SIZE = 16384
EVAL_BATCH_SIZE = 131072
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
MIN_VALID_LOSS_DELTA = 1e-5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
HIDDEN_SIZES = [256, 128, 64]
EMBEDDING_DIM_CAP = 32
TARGET_NAME = "EU"
USE_CONFIDENCE_WEIGHTS = True
MIN_TARGET_WEIGHT = 0.05

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

required_paths = [
    GNN_DATA_PATH,
    EU_TARGET_TENSOR_PATH,
    EU_TARGET_TABLE_PATH,
    KUAIRAND_FEATURE_MANIFEST_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing))

print("device:", DEVICE)
print("KuaiRec tensor package:", GNN_DATA_PATH)
print("EU target tensor:", EU_TARGET_TENSOR_PATH)
print("KuaiRand feature manifest:", KUAIRAND_FEATURE_MANIFEST_PATH)
print("outputs:", OUTPUT_DIR)

device: mps
KuaiRec tensor package: /Users/haozhangao/Desktop/RecSys Research/KuaiRec 2.0/data/processed/gnn_data.pt
EU target tensor: /Users/haozhangao/Desktop/RecSys Research/python_code_new/outputs/gnn_eu_targets.pt
KuaiRand feature manifest: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/outputs/feature_build/kuairand_random_prediction_feature_manifest.json
outputs: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs


In [2]:
# Notebook-03 head model diagnostics. This model is not used here; the check is only a quick
# sanity readout for the shared KuaiRec-to-KuaiRand feature contract.
HEAD_METRICS_PATH = OUTPUT_DIR / "kuairec_transfer_head_mlp_kuairand_features_metrics.json"
if HEAD_METRICS_PATH.exists():
    head_metrics = json.loads(HEAD_METRICS_PATH.read_text())
    compact_head_metrics = {
        "best_epoch": head_metrics.get("best_epoch"),
        "best_validation_bce_loss": head_metrics.get("best_validation_bce_loss"),
        "validation_mean_auc": head_metrics.get("validation_metrics", {}).get("mean_auc"),
        "test_mean_auc": head_metrics.get("test_metrics", {}).get("mean_auc"),
        "validation_auc_by_head": head_metrics.get("validation_metrics", {}).get("auc_by_head"),
        "test_auc_by_head": head_metrics.get("test_metrics", {}).get("auc_by_head"),
    }
    print(json.dumps(compact_head_metrics, indent=2))
else:
    print("Notebook-03 head metrics not found. Continuing with EU training.")

{
  "best_epoch": 19,
  "best_validation_bce_loss": 0.31156918014368445,
  "validation_mean_auc": 0.8585965721891217,
  "test_mean_auc": 0.834477088759206,
  "validation_auc_by_head": {
    "y_complete": 0.8581172782118116,
    "y_long": 0.8376079035083311,
    "y_rewatch": 0.8666293519597751,
    "y_neg": 0.8720317550765686
  },
  "test_auc_by_head": {
    "y_complete": 0.8392543430365335,
    "y_long": 0.8110536942287578,
    "y_rewatch": 0.825719298433628,
    "y_neg": 0.8618810193379051
  }
}


In [3]:
manifest = json.loads(KUAIRAND_FEATURE_MANIFEST_PATH.read_text())
label_cols = [TARGET_NAME]
prediction_feature_cols = list(manifest["prediction_feature_cols"])

continuous_cols = [c for c in manifest["continuous_cols"] if c in prediction_feature_cols]
binary_cols = [c for c in manifest["binary_cols"] if c in prediction_feature_cols]
categorical_cols = [c for c in manifest["categorical_cols"] if c in prediction_feature_cols]
feature_cols = continuous_cols + binary_cols + categorical_cols

if feature_cols != prediction_feature_cols:
    raise ValueError("Feature block ordering does not match manifest['prediction_feature_cols']")

forbidden = set(manifest.get("label_cols", [])) | {
    "watch_ratio", "play_duration", "play_time_ms", "duration_ms", "duration_for_ratio_ms",
    "i_video_tag_id", "i_video_tag_name", "EU", "EU_hat", "q_target", "u_star",
    "score", "score_vanilla", "pred_EU", "pred_q_target",
}
leaked = sorted(set(feature_cols) & forbidden)
if leaked:
    raise ValueError(f"Forbidden columns are in the feature set: {leaked}")

print("target:", TARGET_NAME)
print("continuous:", len(continuous_cols))
print("binary:", len(binary_cols))
print("categorical:", len(categorical_cols))
print("total features:", len(feature_cols))
print("dropped unsafe features in manifest:", manifest.get("dropped_unsafe_features"))
print("model-only dropped features:", sorted(set(manifest["model_feature_cols"]) - set(prediction_feature_cols)))

display(pd.DataFrame({
    "feature": feature_cols,
    "type": (["continuous"] * len(continuous_cols)) + (["binary"] * len(binary_cols)) + (["categorical"] * len(categorical_cols)),
}))

target: EU
continuous: 22
binary: 6
categorical: 33
total features: 61
dropped unsafe features in manifest: ['i_video_tag_id', 'i_video_tag_name']
model-only dropped features: ['hist_author_recency_days']


,feature,type
0,u_follow_user_num,continuous
1,u_fans_user_num,continuous
2,u_friend_user_num,continuous
3,u_register_days,continuous
4,u_follow_user_num_log1p,continuous
...,...,...
56,i_visible_status,categorical
57,i_music_id,categorical
58,i_cat_level1_id,categorical
59,i_cat_level2_id,categorical


In [4]:
gnn_data = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
eu_targets = torch.load(EU_TARGET_TENSOR_PATH, map_location="cpu", weights_only=False)
metadata = gnn_data["feature_metadata"]

source_continuous_cols = list(metadata["continuous_cols"])
source_binary_cols = list(metadata["binary_cols"])
source_categorical_cols = list(metadata["categorical_cols"])

missing_cont = sorted(set(continuous_cols) - set(source_continuous_cols))
missing_bin = sorted(set(binary_cols) - set(source_binary_cols))
missing_cat = sorted(set(categorical_cols) - set(source_categorical_cols))
if missing_cont or missing_bin or missing_cat:
    raise KeyError({
        "missing_continuous": missing_cont,
        "missing_binary": missing_bin,
        "missing_categorical": missing_cat,
    })

cont_idx = torch.tensor([source_continuous_cols.index(c) for c in continuous_cols], dtype=torch.long)
bin_idx = torch.tensor([source_binary_cols.index(c) for c in binary_cols], dtype=torch.long)
cat_idx = torch.tensor([source_categorical_cols.index(c) for c in categorical_cols], dtype=torch.long)

all_categorical_cardinalities = list(metadata["categorical_cardinalities"])
categorical_cardinalities = [
    int(all_categorical_cardinalities[source_categorical_cols.index(c)])
    for c in categorical_cols
]

x_cont_all = gnn_data["x_cont"].cpu()
x_bin_all = gnn_data["x_bin"].cpu()
x_cat_all = gnn_data["x_cat"].cpu()
train_idx_all = gnn_data["train_idx"].cpu().long()
val_idx_all = gnn_data["val_idx"].cpu().long()
test_idx_all = gnn_data["test_idx"].cpu().long()

if TARGET_NAME == "EU":
    y_all = eu_targets["y_EU"].cpu().float()
elif TARGET_NAME == "q_target":
    y_all = eu_targets["y_q_target"].cpu().float()
else:
    raise ValueError(f"Unsupported TARGET_NAME: {TARGET_NAME}")

target_mask = eu_targets["target_mask"].cpu().bool()
num_edges = int(x_cont_all.shape[0])
if len(target_mask) != num_edges:
    raise ValueError(f"target_mask length {len(target_mask)} does not match num_edges {num_edges}")
if not torch.isfinite(y_all[target_mask]).all():
    raise ValueError("Non-finite EU target values on target_mask")

confidence_weight_all = eu_targets.get("target_weight_confidence")
if confidence_weight_all is None:
    confidence_weight_all = torch.ones(num_edges, dtype=torch.float32)
else:
    confidence_weight_all = confidence_weight_all.cpu().float()
confidence_weight_all = torch.where(
    torch.isfinite(confidence_weight_all),
    confidence_weight_all,
    torch.zeros_like(confidence_weight_all),
).clamp(min=float(MIN_TARGET_WEIGHT), max=1.0)
if not USE_CONFIDENCE_WEIGHTS:
    confidence_weight_all = torch.ones_like(confidence_weight_all)

split_edge_ids = {
    "train": train_idx_all[target_mask.index_select(0, train_idx_all)],
    "validation": val_idx_all[target_mask.index_select(0, val_idx_all)],
    "test": test_idx_all[target_mask.index_select(0, test_idx_all)],
}

split_summary = pd.DataFrame({
    "split": list(split_edge_ids.keys()),
    "rows": [int(v.numel()) for v in split_edge_ids.values()],
})
split_summary["share"] = split_summary["rows"] / split_summary["rows"].sum()

train_target = y_all.index_select(0, split_edge_ids["train"])
target_mean = float(train_target.mean())
target_std = float(train_target.std().clamp_min(1e-6))

print("source edge count:", num_edges)
print("EU-labeled rows:", int(target_mask.sum()))
display(split_summary)
print("target mean/std from train split:", target_mean, target_std)

target_rows = []
for split_name, edge_ids in split_edge_ids.items():
    vals = y_all.index_select(0, edge_ids).numpy()
    target_rows.append({
        "split": split_name,
        "target": TARGET_NAME,
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "p01": float(np.quantile(vals, 0.01)),
        "p05": float(np.quantile(vals, 0.05)),
        "p50": float(np.quantile(vals, 0.50)),
        "p95": float(np.quantile(vals, 0.95)),
        "p99": float(np.quantile(vals, 0.99)),
    })
display(pd.DataFrame(target_rows))

print("x_cont_all:", tuple(x_cont_all.shape), "selected:", len(cont_idx))
print("x_bin_all:", tuple(x_bin_all.shape), "selected:", len(bin_idx))
print("x_cat_all:", tuple(x_cat_all.shape), "selected:", len(cat_idx))

source edge count: 13326031
EU-labeled rows: 2519551


,split,rows,share
0,train,2201982,0.873958
1,validation,209791,0.083265
2,test,107778,0.042777


target mean/std from train split: 2.2462661266326904 1.9457354545593262


,split,target,mean,std,p01,p05,p50,p95,p99
0,train,EU,2.246267,1.945735,-2.548888,-0.813116,2.507365,4.672122,5.133349
1,validation,EU,2.292085,1.915865,-2.162464,-0.656407,2.531977,4.682058,5.155352
2,test,EU,2.207960,1.880260,-2.110351,-0.805825,2.415188,4.672212,5.101649


x_cont_all: (13326031, 23) selected: 22
x_bin_all: (13326031, 6) selected: 6
x_cat_all: (13326031, 35) selected: 33


In [5]:
def make_loader(edge_ids, batch_size, shuffle):
    ds = TensorDataset(edge_ids.cpu().long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False)


def fetch_batch(edge_ids):
    edge_ids = edge_ids.cpu().long()
    xb_cont = x_cont_all.index_select(0, edge_ids).index_select(1, cont_idx).float()
    xb_bin = x_bin_all.index_select(0, edge_ids).index_select(1, bin_idx).float()
    xb_cat = x_cat_all.index_select(0, edge_ids).index_select(1, cat_idx).long()
    y_raw = y_all.index_select(0, edge_ids).float().unsqueeze(1)
    y_std = ((y_raw - target_mean) / target_std).float()
    weights = confidence_weight_all.index_select(0, edge_ids).float().unsqueeze(1)
    return xb_cont.to(DEVICE), xb_bin.to(DEVICE), xb_cat.to(DEVICE), y_std.to(DEVICE), y_raw.to(DEVICE), weights.to(DEVICE)

sample_batch = next(iter(make_loader(split_edge_ids["train"], BATCH_SIZE, shuffle=True)))[0]
xb_cont, xb_bin, xb_cat, y_std, y_raw, weights = fetch_batch(sample_batch)
print("sample batch shapes:", tuple(xb_cont.shape), tuple(xb_bin.shape), tuple(xb_cat.shape), tuple(y_std.shape), tuple(weights.shape))
print("sample target mean/std:", float(y_raw.mean().detach().cpu()), float(y_raw.std().detach().cpu()))

sample batch shapes: (16384, 22) (16384, 6) (16384, 33) (16384, 1) (16384, 1)
sample target mean/std: 2.2407898902893066 1.9283939599990845


## MLP Structure

The model mirrors notebook 03's edge-feature MLP, but with one regression output instead of four binary heads.

- Continuous block: standardized KuaiRec edge features.
- Binary block: 0/1 edge features.
- Categorical block: embedding lookup per categorical feature.
- MLP body: `[256, 128, 64]` hidden units with SiLU activations and dropout.
- Loss: weighted MSE on train-standardized EU.

In [6]:
def default_embedding_dim(cardinality, cap=EMBEDDING_DIM_CAP):
    return min(int(cap), max(4, int(round(math.sqrt(int(cardinality))))))


class EdgeFeatureEncoder(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, emb_dim_cap=32):
        super().__init__()
        self.n_cont = int(n_cont)
        self.n_bin = int(n_bin)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]
        self.embedding_dims = [default_embedding_dim(c, emb_dim_cap) for c in self.categorical_cardinalities]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])
        self.output_dim = self.n_cont + self.n_bin + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.n_cont:
            parts.append(x_cont_batch.float())
        if self.n_bin:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=1)


class TransferEUMLP(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, hidden_sizes, dropout=0.1):
        super().__init__()
        self.encoder = EdgeFeatureEncoder(n_cont, n_bin, categorical_cardinalities, EMBEDDING_DIM_CAP)
        layers = []
        dim = self.encoder.output_dim
        for hidden in hidden_sizes:
            layers.append(nn.Linear(dim, int(hidden)))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(float(dropout)))
            dim = int(hidden)
        layers.append(nn.Linear(dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        return self.mlp(self.encoder(x_cont_batch, x_bin_batch, x_cat_batch))


model = TransferEUMLP(
    n_cont=len(continuous_cols),
    n_bin=len(binary_cols),
    categorical_cardinalities=categorical_cardinalities,
    hidden_sizes=HIDDEN_SIZES,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)
print("encoder output dim:", model.encoder.output_dim)
print("first 10 embedding dims:", model.encoder.embedding_dims[:10])
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

TransferEUMLP(
  (encoder): EdgeFeatureEncoder(
    (embeddings): ModuleList(
      (0): Embedding(4, 4)
      (1): Embedding(242, 16)
      (2): Embedding(5, 4)
      (3): Embedding(9, 4)
      (4-6): 3 x Embedding(8, 4)
      (7): Embedding(3, 4)
      (8): Embedding(9, 4)
      (9): Embedding(31, 6)
      (10): Embedding(1077, 32)
      (11): Embedding(14, 4)
      (12): Embedding(11, 4)
      (13): Embedding(4, 4)
      (14): Embedding(48, 7)
      (15): Embedding(341, 18)
      (16): Embedding(8, 4)
      (17): Embedding(6, 4)
      (18-24): 7 x Embedding(4, 4)
      (25): Embedding(7434, 32)
      (26): Embedding(3, 4)
      (27): Embedding(19, 4)
      (28): Embedding(3, 4)
      (29): Embedding(7686, 32)
      (30): Embedding(40, 6)
      (31): Embedding(138, 12)
      (32): Embedding(215, 15)
    )
  )
  (mlp): Sequential(
    (0): Linear(in_features=296, out_features=256, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_fe

In [7]:
def weighted_mse(pred_std, target_std, weights):
    err2 = (pred_std - target_std).square()
    weights = weights.float().clamp_min(1e-8)
    return (err2 * weights).sum() / weights.sum()


def safe_corr(a, b, method="pearson"):
    a = pd.Series(np.asarray(a, dtype=np.float64))
    b = pd.Series(np.asarray(b, dtype=np.float64))
    mask = a.notna() & b.notna()
    if mask.sum() < 3 or a[mask].std(ddof=0) <= 0 or b[mask].std(ddof=0) <= 0:
        return np.nan
    return float(a[mask].corr(b[mask], method=method))


@torch.no_grad()
def evaluate_model(loader, split_name):
    model.eval()
    total_loss = 0.0
    total_weight = 0.0
    pred_chunks = []
    target_chunks = []
    weight_chunks = []

    for (edge_ids,) in loader:
        xb_cont, xb_bin, xb_cat, y_std, y_raw, weights = fetch_batch(edge_ids)
        pred_std = model(xb_cont, xb_bin, xb_cat)
        loss = weighted_mse(pred_std, y_std, weights)
        batch_weight = float(weights.detach().cpu().sum())
        total_loss += float(loss.detach().cpu()) * batch_weight
        total_weight += batch_weight
        pred_raw = pred_std.detach().cpu().squeeze(1) * target_std + target_mean
        pred_chunks.append(pred_raw)
        target_chunks.append(y_raw.detach().cpu().squeeze(1))
        weight_chunks.append(weights.detach().cpu().squeeze(1))

    pred = torch.cat(pred_chunks).numpy()
    target = torch.cat(target_chunks).numpy()
    weights_np = torch.cat(weight_chunks).numpy()
    err = pred - target
    return {
        "split": split_name,
        "loss": total_loss / max(total_weight, 1e-8),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "pearson": safe_corr(pred, target, method="pearson"),
        "spearman": safe_corr(pred, target, method="spearman"),
        "prediction_mean": float(np.mean(pred)),
        "prediction_std": float(np.std(pred)),
        "target_mean": float(np.mean(target)),
        "target_std": float(np.std(target)),
        "mean_weight": float(np.mean(weights_np)),
        "rows": int(len(target)),
    }


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    total_weight = 0.0
    for (edge_ids,) in loader:
        xb_cont, xb_bin, xb_cat, y_std, _, weights = fetch_batch(edge_ids)
        optimizer.zero_grad(set_to_none=True)
        pred_std = model(xb_cont, xb_bin, xb_cat)
        loss = weighted_mse(pred_std, y_std, weights)
        loss.backward()
        optimizer.step()
        batch_weight = float(weights.detach().cpu().sum())
        total_loss += float(loss.detach().cpu()) * batch_weight
        total_weight += batch_weight
    return total_loss / max(total_weight, 1e-8)

In [8]:
train_loader = make_loader(split_edge_ids["train"], BATCH_SIZE, shuffle=True)
val_loader = make_loader(split_edge_ids["validation"], EVAL_BATCH_SIZE, shuffle=False)
test_loader = make_loader(split_edge_ids["test"], EVAL_BATCH_SIZE, shuffle=False)

training_history = []
best_state = None
best_valid_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False
patience_enabled = EARLY_STOPPING_PATIENCE is not None and int(EARLY_STOPPING_PATIENCE) > 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(train_loader)
    valid_metrics = evaluate_model(val_loader, "validation")

    improved = valid_metrics["loss"] < (best_valid_loss - MIN_VALID_LOSS_DELTA)
    if improved:
        best_valid_loss = float(valid_metrics["loss"])
        best_epoch = int(epoch)
        epochs_without_improvement = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    record = {
        "epoch": int(epoch),
        "train_loss": float(train_loss),
        "valid_loss": float(valid_metrics["loss"]),
        "valid_mae": valid_metrics["mae"],
        "valid_rmse": valid_metrics["rmse"],
        "valid_pearson": valid_metrics["pearson"],
        "valid_spearman": valid_metrics["spearman"],
        "best_valid_loss": float(best_valid_loss),
        "best_epoch": int(best_epoch),
        "improved": bool(improved),
        "epochs_without_improvement": int(epochs_without_improvement),
    }
    training_history.append(record)
    print(record)

    if patience_enabled and epochs_without_improvement >= int(EARLY_STOPPING_PATIENCE):
        early_stopped = True
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch={best_epoch}, best validation standardized MSE={best_valid_loss:.6f}, "
            f"patience={int(EARLY_STOPPING_PATIENCE)}"
        )
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

history_df = pd.DataFrame(training_history)
display(history_df.tail())
print("best epoch:", best_epoch)
print("best validation loss:", best_valid_loss)

{'epoch': 1, 'train_loss': 0.38016764323680113, 'valid_loss': 0.1586620769069598, 'valid_mae': 0.5862705111503601, 'valid_rmse': 0.7628946900367737, 'valid_pearson': 0.9186248636063821, 'valid_spearman': 0.9003865706393703, 'best_valid_loss': 0.1586620769069598, 'best_epoch': 1, 'improved': True, 'epochs_without_improvement': 0}
{'epoch': 2, 'train_loss': 0.13099950666921245, 'valid_loss': 0.1010829022091806, 'valid_mae': 0.44505661725997925, 'valid_rmse': 0.6083002686500549, 'valid_pearson': 0.9484397390418754, 'valid_spearman': 0.938781150912727, 'best_valid_loss': 0.1010829022091806, 'best_epoch': 2, 'improved': True, 'epochs_without_improvement': 0}
{'epoch': 3, 'train_loss': 0.10417906250614924, 'valid_loss': 0.08986060027600949, 'valid_mae': 0.41873490810394287, 'valid_rmse': 0.5757253766059875, 'valid_pearson': 0.954125014569263, 'valid_spearman': 0.9458865937321169, 'best_valid_loss': 0.08986060027600949, 'best_epoch': 3, 'improved': True, 'epochs_without_improvement': 0}
{'epo

,epoch,train_loss,valid_loss,valid_mae,valid_rmse,valid_pearson,valid_spearman,best_valid_loss,best_epoch,improved,epochs_without_improvement
25,26,0.067049,0.063432,0.352496,0.498192,0.966092,0.960279,0.063379,25,False,1
26,27,0.066805,0.063140,0.347694,0.493462,0.966532,0.960575,0.063140,27,True,0
27,28,0.066430,0.063031,0.356878,0.494921,0.966505,0.960698,0.063031,28,True,0
28,29,0.066146,0.062649,0.344870,0.493082,0.966667,0.960876,0.062649,29,True,0
29,30,0.065905,0.062442,0.349435,0.493937,0.966680,0.960980,0.062442,30,True,0


best epoch: 30
best validation loss: 0.06244230728377426


In [9]:
final_train_metrics = evaluate_model(train_loader, "train")
final_valid_metrics = evaluate_model(val_loader, "validation")
final_test_metrics = evaluate_model(test_loader, "test")

print("train metrics")
print(json.dumps(final_train_metrics, indent=2))
print("validation metrics")
print(json.dumps(final_valid_metrics, indent=2))
print("test metrics")
print(json.dumps(final_test_metrics, indent=2))

display(pd.DataFrame([final_train_metrics, final_valid_metrics, final_test_metrics]))

train metrics
{
  "split": "train",
  "loss": 0.05861031614456118,
  "mae": 0.3377758264541626,
  "rmse": 0.48122647404670715,
  "pearson": 0.9693585568620697,
  "spearman": 0.9644478361421789,
  "prediction_mean": 2.210801124572754,
  "prediction_std": 1.9293100833892822,
  "target_mean": 2.246265172958374,
  "target_std": 1.945734977722168,
  "mean_weight": 0.8272339105606079,
  "rows": 2201982
}
validation metrics
{
  "split": "validation",
  "loss": 0.06244230728377426,
  "mae": 0.3494347035884857,
  "rmse": 0.4939367175102234,
  "pearson": 0.9666798569321717,
  "spearman": 0.9609801741012853,
  "prediction_mean": 2.2527923583984375,
  "prediction_std": 1.8956060409545898,
  "target_mean": 2.2920851707458496,
  "target_std": 1.9158645868301392,
  "mean_weight": 0.8259503841400146,
  "rows": 209791
}
test metrics
{
  "split": "test",
  "loss": 0.06454741954803467,
  "mae": 0.3588150143623352,
  "rmse": 0.5042319893836975,
  "pearson": 0.9640672461030203,
  "spearman": 0.960931975487

,split,loss,mae,rmse,pearson,spearman,prediction_mean,prediction_std,target_mean,target_std,mean_weight,rows
0,train,0.058610,0.337776,0.481226,0.969359,0.964448,2.210801,1.929310,2.246265,1.945735,0.827234,2201982
1,validation,0.062442,0.349435,0.493937,0.966680,0.960980,2.252792,1.895606,2.292085,1.915865,0.825950,209791
2,test,0.064547,0.358815,0.504232,0.964067,0.960932,2.161741,1.863752,2.207960,1.880260,0.827204,107778


## Save Transfer EU Predictor Artifacts

The saved model bundle contains the trained weights, feature contract, KuaiRec preprocessing metadata, and target standardization constants. A later notebook can load this bundle and apply it to KuaiRand random rows.

In [10]:
def selected_dict(source, keys):
    return {k: source[k] for k in keys if k in source}

selected_feature_metadata = {
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "continuous_scaler": selected_dict(metadata.get("continuous_scaler", {}), continuous_cols),
    "continuous_missing": selected_dict(metadata.get("continuous_missing", {}), continuous_cols),
    "binary_missing": selected_dict(metadata.get("binary_missing", {}), binary_cols),
    "categorical_mappings": selected_dict(metadata.get("categorical_mappings", {}), categorical_cols),
    "categorical_cardinalities": categorical_cardinalities,
    "categorical_encoding": metadata.get("categorical_encoding"),
    "max_cardinality_without_hashing": metadata.get("max_cardinality_without_hashing"),
    "hash_bucket_size": metadata.get("hash_bucket_size"),
    "unknown_code": 0,
}

feature_spec = {
    "run_name": RUN_NAME,
    "trained_on": "KuaiRec",
    "intended_scoring_data": "KuaiRand random recommendations",
    "source_tensor_package": str(GNN_DATA_PATH),
    "source_processed_table": str(KUAIREC_SOURCE_TABLE_PATH),
    "source_eu_target_tensor": str(EU_TARGET_TENSOR_PATH),
    "source_eu_target_table": str(EU_TARGET_TABLE_PATH),
    "kuairand_manifest_path": str(KUAIRAND_FEATURE_MANIFEST_PATH),
    "feature_contract": "manifest['prediction_feature_cols']",
    "target_name": TARGET_NAME,
    "target_standardization": {
        "mean": float(target_mean),
        "std": float(target_std),
        "fit_split": "KuaiRec train rows with notebook-07 EU targets",
    },
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "feature_cols": feature_cols,
    "dropped_unsafe_features": manifest.get("dropped_unsafe_features", []),
    "model_only_features_not_used": sorted(set(manifest["model_feature_cols"]) - set(prediction_feature_cols)),
    "architecture": {
        "model_class": "TransferEUMLP",
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
        "embedding_dim_rule": "min(32, max(4, round(sqrt(cardinality))))",
        "embedding_dims": model.encoder.embedding_dims,
        "encoder_output_dim": int(model.encoder.output_dim),
        "output_dim": 1,
        "loss": f"weighted MSE over train-standardized {TARGET_NAME}",
    },
    "selected_feature_metadata": selected_feature_metadata,
}

metrics = {
    "run_name": RUN_NAME,
    "random_seed": RANDOM_SEED,
    "device": DEVICE,
    "use_tiny": bool(USE_TINY),
    "target_name": TARGET_NAME,
    "num_epochs_requested": NUM_EPOCHS,
    "num_epochs_run": len(training_history),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_enabled": bool(patience_enabled),
    "min_valid_loss_delta": MIN_VALID_LOSS_DELTA,
    "early_stopped": bool(early_stopped),
    "best_epoch": int(best_epoch),
    "best_validation_loss": float(best_valid_loss),
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "hidden_sizes": HIDDEN_SIZES,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "use_confidence_weights": bool(USE_CONFIDENCE_WEIGHTS),
    "min_target_weight": float(MIN_TARGET_WEIGHT),
    "split_rows": split_summary.to_dict(orient="records"),
    "train_metrics": final_train_metrics,
    "validation_metrics": final_valid_metrics,
    "test_metrics": final_test_metrics,
    "feature_spec_path": str(FEATURE_SPEC_OUTPUT_PATH),
    "model_output_path": str(MODEL_OUTPUT_PATH),
    "history_output_path": str(HISTORY_OUTPUT_PATH),
    "metrics_output_path": str(METRICS_OUTPUT_PATH),
    "feature_cols": feature_cols,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
}

history_df.to_csv(HISTORY_OUTPUT_PATH, index=False)
FEATURE_SPEC_OUTPUT_PATH.write_text(json.dumps(feature_spec, indent=2, default=str) + "\n")
METRICS_OUTPUT_PATH.write_text(json.dumps(metrics, indent=2, default=str) + "\n")

torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "model_class": "TransferEUMLP",
    "config": feature_spec["architecture"],
    "feature_spec": feature_spec,
    "metrics": metrics,
}, MODEL_OUTPUT_PATH)

print("saved model:", MODEL_OUTPUT_PATH)
print("saved metrics:", METRICS_OUTPUT_PATH)
print("saved history:", HISTORY_OUTPUT_PATH)
print("saved feature spec:", FEATURE_SPEC_OUTPUT_PATH)

saved model: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_eu_mlp_kuairand_features.pt
saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_eu_mlp_kuairand_features_metrics.json
saved history: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_eu_mlp_kuairand_features_history.csv
saved feature spec: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_eu_mlp_kuairand_features_feature_spec.json


## Next Step

After this notebook finishes, use a separate notebook to load the saved EU predictor and score KuaiRand random recommendations. Keeping training and KuaiRand inference separate makes it easier to verify that the EU predictor itself is trained only on KuaiRec.